In [0]:
%sql
CREATE DATABASE IF NOT EXISTS googleads_analytics;

In [0]:
%sql
CREATE OR REPLACE VIEW googleads_analytics.vw_campaign_performance AS
SELECT
    c.campaign_name,
    c.advertising_channel_type,
    f.segments_date,
    f.impressions,
    f.clicks,
    f.conversions,
    f.cost_inr,
    f.ctr_pct,
    f.cpc_inr,
    f.cvr_pct,
    f.cpa_inr
FROM googleads_gold.fact_campaign_perf f
JOIN googleads_gold.dim_campaign c
ON f.campaign_pk = c.campaign_pk;

In [0]:
%sql
CREATE OR REPLACE VIEW googleads_analytics.vw_keyword_performance AS
SELECT
    c.campaign_name,
    ag.ad_group_name,
    k.keyword,
    k.keyword_match_type,
    f.segments_date,
    f.impressions,
    f.clicks,
    f.conversions,
    f.cost_inr
FROM googleads_gold.fact_keyword_perf f
JOIN googleads_gold.dim_campaign c
ON f.campaign_pk = c.campaign_pk
JOIN googleads_gold.dim_ad_group ag
ON f.ad_group_pk = ag.ad_group_pk
JOIN googleads_gold.dim_keyword k
ON f.keyword_pk = k.keyword_pk;

In [0]:
%sql
CREATE OR REPLACE VIEW googleads_analytics.vw_search_term_performance AS
SELECT
    c.campaign_name,
    ag.ad_group_name,
    st.search_term,
    f.segments_date,
    f.impressions,
    f.clicks,
    f.conversions,
    f.cost_inr
FROM googleads_gold.fact_search_term_perf f
JOIN googleads_gold.dim_campaign c
ON f.campaign_pk = c.campaign_pk
JOIN googleads_gold.dim_ad_group ag
ON f.ad_group_pk = ag.ad_group_pk
JOIN googleads_gold.dim_search_term st
ON f.search_term_pk = st.search_term_pk;

In [0]:
from pyspark.sql import functions as F

fact_campaign_perf = spark.table(
    "googleads_gold.fact_campaign_perf"
)

campaign_summary = fact_campaign_perf.groupBy(
    "campaign_pk"
).agg(
    F.sum("impressions").alias("impressions"),
    F.sum("clicks").alias("clicks"),
    F.sum("conversions").alias("conversions"),
    F.sum("cost_inr").alias("cost_inr")
)

display(campaign_summary)

campaign_pk,impressions,clicks,conversions,cost_inr
f91c8857e9fdba0db482390b1c04225393d46f6ca602c689915784b34694f37f,41441,4763,0.0,1545.775615
9bad51fc0227ca9dd34d0ce0b981e1c044635cd7102c2da9a390a0c9c0f19287,133103,6269,2.0,16952.824176000006
290bea4e7806018de53d3d8f630fd55311a5c53a6cfe2f2a36c591a56412f5b1,90137,6905,0.0,6984.6457310000005
6526a2be9f28372b0e5c4f6a61004d7897cea55bcc89efabb92a36f24c11651b,53903,444,0.0,18755.628474999998
412506d2e547c9170ca121e090fcae4d817da3e0beca8371bcd342a34a6e9b83,23199,1031,253.0,29311.538660999995
6d9ce7c4c5076914f74593fe56e879cd0a7cd634b5043bdb4a9e47cea8da9446,85853,8134,849.0,6732.6109289999995
e4c032e7fd36b3a09a0793617a846325235e50ec91a18e233924f198f5330b18,14999,768,195.0,25152.655914999992
fa7b63b93ed5ba4ecc5cff4391cea1ee258e167fd085f3ab9b6f9d13e37e7517,2826,139,0.0,541.2477880000001


In [0]:
campaign_summary.write.format("delta") \
.mode("overwrite") \
.saveAsTable("googleads_analytics.campaign_summary")

In [0]:
from pyspark.sql import functions as F

fact_device_perf = spark.table(
    "googleads_gold.fact_device_perf"
)

device_summary = fact_device_perf.groupBy(
    "device_pk"
).agg(
    F.sum("impressions").alias("impressions"),
    F.sum("clicks").alias("clicks"),
    F.sum("conversions").alias("conversions"),
    F.sum("cost_inr").alias("cost_inr")
)

device_summary.write.format("delta") \
.mode("overwrite") \
.saveAsTable("googleads_analytics.device_summary")

In [0]:
fact_keyword_perf = spark.table(
    "googleads_gold.fact_keyword_perf"
)

keyword_summary = fact_keyword_perf.groupBy(
    "keyword_pk"
).agg(
    F.sum("impressions").alias("impressions"),
    F.sum("clicks").alias("clicks"),
    F.sum("conversions").alias("conversions"),
    F.sum("cost_inr").alias("cost_inr")
)

keyword_summary.write.format("delta") \
.mode("overwrite") \
.saveAsTable("googleads_analytics.keyword_summary")

In [0]:
fact_search_term_perf = spark.table(
    "googleads_gold.fact_search_term_perf"
)

search_term_summary = fact_search_term_perf.groupBy(
    "search_term_pk"
).agg(
    F.sum("impressions").alias("impressions"),
    F.sum("clicks").alias("clicks"),
    F.sum("conversions").alias("conversions"),
    F.sum("cost_inr").alias("cost_inr")
)

search_term_summary.write.format("delta") \
.mode("overwrite") \
.saveAsTable("googleads_analytics.search_term_summary")

In [0]:
%sql
SELECT
    c.campaign_name,
    b.budget_inr,
    SUM(f.cost_inr) AS spend
FROM googleads_gold.fact_campaign_perf f
JOIN googleads_gold.dim_campaign c
    ON f.campaign_pk = c.campaign_pk
JOIN googleads_gold.dim_budget b
    ON c.budget_pk = b.budget_pk
GROUP BY
    c.campaign_name,
    b.budget_inr;

campaign_name,budget_inr,spend
Data Engineering (AWS-Sept),1003.5,6984.6457310000005
10-Sept AWS Snowflake,1050.0,18755.628474999998
may 21 data engineering,500.0,541.2477880000001
Azure Data Engineering -Display - Image,500.0,1545.775615
APRIL 21ST,500.0,25891.784550999993
april 21st,500.0,6739.729646999999
Azure-Data-Engineering-Jan-2026,500.0,29311.538660999995
Hyderabad,1500.0,16853.478501000005


In [0]:
fact_landing_page_perf = spark.table(
    "googleads_gold.fact_landing_page_perf"
)

landing_page_summary = fact_landing_page_perf.groupBy(
    "landing_page_pk"
).agg(
    F.sum("impressions").alias("impressions"),
    F.sum("clicks").alias("clicks"),
    F.sum("conversions").alias("conversions"),
    F.sum("cost_inr").alias("cost_inr")
)

landing_page_summary.write.format("delta") \
.mode("overwrite") \
.saveAsTable("googleads_analytics.landing_page_summary")

In [0]:
fact_geo_perf = spark.table(
    "googleads_gold.fact_geo_perf"
)

geo_summary = fact_geo_perf.groupBy(
    "location_pk"
).agg(
    F.sum("impressions").alias("impressions"),
    F.sum("clicks").alias("clicks"),
    F.sum("conversions").alias("conversions"),
    F.sum("cost_inr").alias("cost_inr")
)

geo_summary.write.format("delta") \
.mode("overwrite") \
.saveAsTable("googleads_analytics.geo_summary")

In [0]:
fact_age_perf = spark.table(
    "googleads_gold.fact_age_perf"
)

age_summary = fact_age_perf.groupBy(
    "age_range_pk"
).agg(
    F.sum("impressions").alias("impressions"),
    F.sum("clicks").alias("clicks"),
    F.sum("conversions").alias("conversions"),
    F.sum("cost_inr").alias("cost_inr")
)

age_summary.write.format("delta") \
.mode("overwrite") \
.saveAsTable("googleads_analytics.age_summary")

In [0]:
fact_gender_perf = spark.table(
    "googleads_gold.fact_gender_perf"
)

gender_summary = fact_gender_perf.groupBy(
    "gender_pk"
).agg(
    F.sum("impressions").alias("impressions"),
    F.sum("clicks").alias("clicks"),
    F.sum("conversions").alias("conversions"),
    F.sum("cost_inr").alias("cost_inr")
)

gender_summary.write.format("delta") \
.mode("overwrite") \
.saveAsTable("googleads_analytics.gender_summary")

In [0]:
%sql
OPTIMIZE googleads_gold.fact_campaign_perf;

OPTIMIZE googleads_gold.fact_keyword_perf;

OPTIMIZE googleads_gold.fact_search_term_perf;

OPTIMIZE googleads_gold.fact_device_perf;

OPTIMIZE googleads_gold.fact_landing_page_perf;

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1781503376136, 1781503376669, 8, 0, null, List(0, 0), null, 11, 11, 0, 0, null, null)"


In [0]:
%sql

ANALYZE TABLE googleads_gold.fact_campaign_perf COMPUTE STATISTICS;

ANALYZE TABLE googleads_gold.fact_keyword_perf COMPUTE STATISTICS;

ANALYZE TABLE googleads_gold.fact_search_term_perf COMPUTE STATISTICS;

ANALYZE TABLE googleads_gold.fact_device_perf COMPUTE STATISTICS;

ANALYZE TABLE googleads_gold.fact_landing_page_perf COMPUTE STATISTICS;

In [0]:
%sql

OPTIMIZE googleads_gold.fact_campaign_perf
ZORDER BY (campaign_pk);

OPTIMIZE googleads_gold.fact_keyword_perf
ZORDER BY (campaign_pk, ad_group_pk, keyword_pk);

OPTIMIZE googleads_gold.fact_search_term_perf
ZORDER BY (campaign_pk, ad_group_pk, search_term_pk);

OPTIMIZE googleads_gold.fact_device_perf
ZORDER BY (campaign_pk, device_pk);

OPTIMIZE googleads_gold.fact_landing_page_perf
ZORDER BY (campaign_pk, landing_page_pk);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 19899), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1781503514315, 1781503514967, 8, 0, null, List(0, 0), null, 11, 11, 0, 0, null, null)"


In [0]:
%sql
SHOW TABLES IN googleads_analytics;

database,tableName,isTemporary
googleads_analytics,age_summary,false
googleads_analytics,campaign_summary,false
googleads_analytics,device_summary,false
googleads_analytics,gender_summary,false
googleads_analytics,geo_summary,false
googleads_analytics,keyword_summary,false
googleads_analytics,landing_page_summary,false
googleads_analytics,search_term_summary,false
googleads_analytics,vw_campaign_performance,false
googleads_analytics,vw_keyword_performance,false


In [0]:
tables = spark.catalog.listTables("googleads_analytics")

for table in tables:
    df = spark.table(f"googleads_analytics.{table.name}")
    print(f"{table.name}: {df.count():,} rows")

age_summary: 7 rows
campaign_summary: 8 rows
device_summary: 5 rows
gender_summary: 3 rows
geo_summary: 1 rows
keyword_summary: 129 rows
landing_page_summary: 10 rows
search_term_summary: 20,972 rows
vw_campaign_performance: 281 rows
vw_keyword_performance: 1,485 rows
vw_search_term_performance: 42,460 rows
